# Stage 2 — Missingness handling

Stage 1 already built the "detect and flag" half of this: per-channel sensor
dropout flags + interpolation (`src/features_signal.py`), and a text
`missing_indicator` (`src/features_text.py`). What's left is the half that
only matters once multiple modalities are combined: **modality dropout
during training**, so a fusion model doesn't quietly learn to lean on
whichever modality is easiest (text, per Stage 1's numbers) and then fall
apart on a flight where that modality happens to be missing or weak.

Only `signal` and `text` are droppable here. Tabular telemetry has zero
missingness anywhere in this dataset -- there's no real condition synthetic
tabular dropout would simulate, so it's left untouched.

This notebook:
1. Builds the combined feature panel (tabular + signal + text embedding)
   that Stage 3's fusion model will also consume.
2. Implements dropout at the *raw* level (zero the signal tensor; blank the
   note) and reruns the normal feature builders, so a synthetically-dropped
   row is indistinguishable from a naturally-missing one.
3. Proves the augmentation does something: two otherwise-identical models,
   one trained with dropout augmentation and one without, evaluated on
   validation data where a modality has been artificially knocked out.


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import lightgbm as lgb

from src.data import load_split
from src.cv import get_folds, assert_disjoint_groups
from src.metrics import auprc, report_folds
from src.feature_panel import build_feature_panel
from src.modality_dropout import apply_modality_dropout

RANDOM_STATE = 42

train = load_split("train")
tab, notes, signals, channel_names = train["tab"], train["notes"], train["signals"], train["channel_names"]
note_text = notes["maintenance_note"]
y = tab["failure_within_horizon"].values
print(f"{len(tab)} flights, {tab['drone_id'].nunique()} drones")


15872 flights, 620 drones


In [2]:
folds = list(get_folds(tab, n_splits=5, seed=RANDOM_STATE))
for tr_idx, val_idx in folds:
    assert_disjoint_groups(tab, tr_idx, val_idx)
print(f"{len(folds)} folds, same split structure as Stage 1 (same seed, same CV scheme)")


5 folds, same split structure as Stage 1 (same seed, same CV scheme)


## Build the clean panel, plus two globally-degraded panels for evaluation

`panel_clean` is the real feature panel. `panel_text_blank` and
`panel_signal_blank` are the *same* panel but with every row's text (resp.
signal) forced to the "missing" representation -- these exist purely so we
can slice out a fold's validation rows and ask "how does this model do if
notes/sensors weren't filed for this batch of flights?" They are never used
for training, only for evaluation.


In [3]:
panel_clean, cat_cols, text_cols = build_feature_panel(tab, note_text, signals, channel_names)
feature_cols = [c for c in panel_clean.columns if c != "flight_id"]
print("panel:", panel_clean.shape)

all_blank_text = pd.Series([""] * len(note_text), index=note_text.index)
panel_text_blank, _, _ = build_feature_panel(tab, all_blank_text, signals, channel_names)

all_blank_signal = np.zeros_like(signals)
panel_signal_blank, _, _ = build_feature_panel(tab, note_text, all_blank_signal, channel_names)


C:\Users\aneja\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


panel: (15872, 1181)


## The ablation

For each fold: train **Model A** (no augmentation) and **Model B** (15%
signal dropout + 15% text dropout applied to that fold's training rows only)
on identical LightGBM settings. Evaluate both on three versions of the same
validation rows: normal, text-missing, and signal-missing.


In [4]:
lgb_params = dict(
    objective="binary",
    n_estimators=400,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbosity=-1,
)

rows = []
for fold_i, (tr_idx, val_idx) in enumerate(folds):
    # --- Model A: no augmentation ---
    model_a = lgb.LGBMClassifier(**lgb_params)
    model_a.fit(panel_clean.loc[tr_idx, feature_cols], y[tr_idx], categorical_feature=cat_cols)

    # --- Model B: trained with modality dropout on this fold's train rows ---
    sig_aug, txt_aug, _, _ = apply_modality_dropout(signals, note_text, tr_idx, p_signal=0.15, p_text=0.15, seed=RANDOM_STATE + fold_i)
    panel_aug, _, _ = build_feature_panel(tab, txt_aug, sig_aug, channel_names)
    model_b = lgb.LGBMClassifier(**lgb_params)
    model_b.fit(panel_aug.loc[tr_idx, feature_cols], y[tr_idx], categorical_feature=cat_cols)

    for name, model in [("A (no dropout aug)", model_a), ("B (with dropout aug)", model_b)]:
        normal = auprc(y[val_idx], model.predict_proba(panel_clean.loc[val_idx, feature_cols])[:, 1])
        text_missing = auprc(y[val_idx], model.predict_proba(panel_text_blank.loc[val_idx, feature_cols])[:, 1])
        signal_missing = auprc(y[val_idx], model.predict_proba(panel_signal_blank.loc[val_idx, feature_cols])[:, 1])
        rows.append({"fold": fold_i, "model": name, "normal": normal, "text_missing": text_missing, "signal_missing": signal_missing})

results = pd.DataFrame(rows)
results


,fold,model,normal,text_missing,signal_missing
0,0,A (no dropout aug),0.533884,0.301833,0.497427
1,0,B (with dropout aug),0.525363,0.322888,0.526595
2,1,A (no dropout aug),0.505643,0.306927,0.490953
3,1,B (with dropout aug),0.501767,0.294419,0.507106
4,2,A (no dropout aug),0.515936,0.307774,0.486280
5,2,B (with dropout aug),0.507003,0.283349,0.509583
6,3,A (no dropout aug),0.533801,0.319746,0.517824
7,3,B (with dropout aug),0.534278,0.320207,0.532545
8,4,A (no dropout aug),0.522713,0.305315,0.462642
9,4,B (with dropout aug),0.525222,0.310923,0.519457


In [5]:
summary = results.groupby("model")[["normal", "text_missing", "signal_missing"]].agg(["mean", "std"])
summary


normal           text_missing            \
                          mean       std         mean       std   
model                                                             
A (no dropout aug)    0.522395  0.012089     0.308319  0.006781   
B (with dropout aug)  0.518727  0.013722     0.306357  0.017013   

                     signal_missing            
                               mean       std  
model                                          
A (no dropout aug)         0.491025  0.019920  
B (with dropout aug)       0.519057  0.010857

## Reading the result

`normal` should be close between A and B -- dropout augmentation shouldn't
cost much when both modalities are actually present, since 70-85% of each
fold's training rows are untouched. The gap to watch is `text_missing` and
`signal_missing`: if B holds up closer to its own `normal` score than A does
to its `normal` score, the augmentation is doing its job -- B learned not to
fully bet on a modality that can disappear.


In [6]:
degradation = results.copy()
degradation["text_drop_cost"] = degradation["normal"] - degradation["text_missing"]
degradation["signal_drop_cost"] = degradation["normal"] - degradation["signal_missing"]
degradation.groupby("model")[["text_drop_cost", "signal_drop_cost"]].mean()


,text_drop_cost,signal_drop_cost
model,,
A (no dropout aug),0.214076,0.031370
B (with dropout aug),0.212369,-0.000331


## Cache the panel for Stage 3

Stage 3's mid-level fusion model reuses this panel directly instead of
rebuilding tabular/signal/text features from scratch.


In [7]:
Path("../artifacts").mkdir(exist_ok=True)
panel_clean.to_parquet("../artifacts/feature_panel_train.parquet")
pd.Series(cat_cols).to_json("../artifacts/feature_panel_cat_cols.json")
print("saved artifacts/feature_panel_train.parquet", panel_clean.shape)


saved artifacts/feature_panel_train.parquet (15872, 1181)
